In [1]:
# ============================================================================
# PART 1: INSTALLATION AND IMPORTS
# ============================================================================

import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchaudio
import torchaudio.transforms as T
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Dict, Optional
import gc
from transformers import ASTFeatureExtractor, ASTModel
import shutil

# Set random seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

set_seed(42)

# Device configuration
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"Found {n_gpus} GPU(s):")
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    n_gpus = 0

print(f"\nUsing device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

Found 2 GPU(s):
  GPU 0: Tesla T4
    Memory: 15.64 GB
  GPU 1: Tesla T4
    Memory: 15.64 GB

Using device: cuda
PyTorch version: 2.9.0+cu126
CUDA version: 12.6


In [2]:
# ============================================================================
# PART 2: CONFIGURATION - OPTIMIZED FOR >0.97 F1
# ============================================================================

class Config:
    # Paths
    BASE_DIR = Path('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup')
    STEMS_DIR = BASE_DIR / 'genres_stems'
    ESC50_DIR = BASE_DIR / 'ESC-50-master' / 'audio'
    MASHUPS_DIR = BASE_DIR / 'mashups'
    TEST_CSV = BASE_DIR / 'test.csv'
    SAMPLE_SUBMISSION = BASE_DIR / 'sample_submission.csv'
    OUTPUT_DIR = Path('/kaggle/working')
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    # Audio Parameters
    SAMPLE_RATE = 16000
    AUDIO_DURATION = 10  # 10 second crops for AST
    NUM_SAMPLES = SAMPLE_RATE * AUDIO_DURATION  # 160,000
    FULL_DURATION = 30  # Full mashup duration
    FULL_SAMPLES = SAMPLE_RATE * FULL_DURATION  # 480,000
    
    # AST Model
    AST_MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
    
    # Mel Spectrogram (AST requirements)
    N_MELS = 128
    N_FFT = 400
    HOP_LENGTH = 160
    FMIN = 20
    FMAX = 8000
    TARGET_LENGTH = 1024
    
    # Training - OPTIMIZED
    BATCH_SIZE = 48 if n_gpus >= 2 else 24
    NUM_EPOCHS = 25
    NUM_WORKERS = 4
    LEARNING_RATE = 5e-6  # Lower for stability
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    GRADIENT_CLIP = 1.0
    
    # Augmentation - REFINED
    SNR_RANGE = (12, 25)
    VOLUME_RANGE = (0.85, 1.15)
    MIXUP_ALPHA = 0.2
    MIXUP_PROB = 0.25
    
    # SpecAugment
    FREQ_MASK_PARAM = 20
    TIME_MASK_PARAM = 40
    NUM_FREQ_MASKS = 2
    NUM_TIME_MASKS = 2
    SPEC_AUGMENT_PROB = 0.5
    
    # TTA - INCREASED
    TTA_CROPS = 9
    
    # Ensemble
    NUM_ENSEMBLE_MODELS = 3
    
    # Genres
    GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
              'jazz', 'metal', 'pop', 'reggae', 'rock']
    NUM_CLASSES = len(GENRES)
    GENRE2IDX = {genre: idx for idx, genre in enumerate(GENRES)}
    IDX2GENRE = {idx: genre for idx, genre in enumerate(GENRES)}
    
    # Dataset
    SYNTHETIC_SAMPLES_PER_EPOCH = 8000
    VAL_SAMPLES = 1500
    
    # Other
    USE_AMP = True
    PATIENCE = 10
    USE_MULTI_GPU = n_gpus >= 2

config = Config()
print(f"\n{'='*60}")
print("CONFIGURATION")
print(f"{'='*60}")
print(f"AST Model: {config.AST_MODEL_NAME}")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Ensemble models: {config.NUM_ENSEMBLE_MODELS}")
print(f"TTA crops: {config.TTA_CROPS}")
print(f"Multi-GPU: {config.USE_MULTI_GPU}")
print(f"{'='*60}")


CONFIGURATION
AST Model: MIT/ast-finetuned-audioset-10-10-0.4593
Batch size: 48
Ensemble models: 3
TTA crops: 9
Multi-GPU: True


In [3]:
# ============================================================================
# PART 3: AUDIO CACHE
# ============================================================================

class AudioCache:
    """Preload all audio into RAM for fast access"""
    
    def __init__(self, stems_dir: Path, esc50_dir: Path, sample_rate: int = 16000):
        self.sr = sample_rate
        self.stems_cache = {}
        self.noise_cache = []
        
        print("\nLoading audio files into RAM...")
        self._load_stems(stems_dir)
        self._load_noise(esc50_dir)
        
        total_stems = self._count_stems()
        print(f"✓ Loaded {total_stems} stems and {len(self.noise_cache)} noise clips")
        
    def _load_audio(self, path: Path, duration: float = None) -> np.ndarray:
        """Load and resample audio"""
        try:
            y, _ = librosa.load(path, sr=self.sr, duration=duration, mono=True)
            return y.astype(np.float32)
        except Exception as e:
            return np.zeros(self.sr * 5, dtype=np.float32)
    
    def _load_stems(self, stems_dir: Path):
        """Load all stems organized by genre"""
        stem_types = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
        
        for genre_dir in tqdm(sorted(stems_dir.iterdir()), desc="Loading stems"):
            if not genre_dir.is_dir():
                continue
            
            genre = genre_dir.name
            if genre not in config.GENRES:
                continue
            
            self.stems_cache[genre] = {}
            
            for song_dir in sorted(genre_dir.iterdir()):
                if not song_dir.is_dir():
                    continue
                
                song_id = song_dir.name
                self.stems_cache[genre][song_id] = {}
                
                for stem_type in stem_types:
                    stem_path = song_dir / stem_type
                    if stem_path.exists():
                        audio = self._load_audio(stem_path, duration=30)
                        self.stems_cache[genre][song_id][stem_type] = audio
    
    def _load_noise(self, esc50_dir: Path):
        """Load ESC-50 noise clips"""
        noise_files = list(esc50_dir.glob('*.wav'))
        
        for noise_path in tqdm(noise_files, desc="Loading noise"):
            audio = self._load_audio(noise_path, duration=5)
            if len(audio) > 0:
                self.noise_cache.append(audio)
    
    def _count_stems(self) -> int:
        count = 0
        for genre in self.stems_cache:
            for song_id in self.stems_cache[genre]:
                count += len(self.stems_cache[genre][song_id])
        return count
    
    def get_random_stems_for_genre(self, genre: str) -> Dict[str, np.ndarray]:
        """Get random stems from different songs of same genre"""
        if genre not in self.stems_cache:
            return {}
        
        song_ids = list(self.stems_cache[genre].keys())
        if len(song_ids) < 4:
            return {}
        
        selected_songs = random.sample(song_ids, min(4, len(song_ids)))
        stem_types = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
        
        stems = {}
        for i, stem_type in enumerate(stem_types):
            song_id = selected_songs[i % len(selected_songs)]
            if stem_type in self.stems_cache[genre][song_id]:
                stems[stem_type] = self.stems_cache[genre][song_id][stem_type].copy()
        
        return stems
    
    def get_random_noise(self) -> np.ndarray:
        """Get random noise clip"""
        if not self.noise_cache:
            return np.zeros(self.sr * 5, dtype=np.float32)
        return random.choice(self.noise_cache).copy()

# Initialize cache
audio_cache = AudioCache(config.STEMS_DIR, config.ESC50_DIR, config.SAMPLE_RATE)
gc.collect()


Loading audio files into RAM...


Loading stems:   0%|          | 0/10 [00:00<?, ?it/s]

Loading noise:   0%|          | 0/2000 [00:00<?, ?it/s]

✓ Loaded 4000 stems and 2000 noise clips


107105

In [4]:
# ============================================================================
# PART 4: AUDIO PROCESSING
# ============================================================================

class AudioProcessor:
    """Audio processing utilities"""
    
    @staticmethod
    def normalize_audio(audio: np.ndarray) -> np.ndarray:
        """Peak normalization"""
        if len(audio) == 0:
            return audio
        max_val = np.abs(audio).max()
        if max_val > 0:
            audio = audio / max_val * 0.95
        return audio
    
    @staticmethod
    def rms_normalize(audio: np.ndarray, target_db: float = -20.0) -> np.ndarray:
        """RMS normalization"""
        if len(audio) == 0:
            return audio
        
        rms = np.sqrt(np.mean(audio ** 2))
        if rms < 1e-8:
            return audio
        
        target_rms = 10 ** (target_db / 20)
        audio = audio * (target_rms / rms)
        audio = np.clip(audio, -1.0, 1.0)
        return audio
    
    @staticmethod
    def add_noise_snr(clean: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
        """Add noise at specified SNR"""
        if len(noise) < len(clean):
            repeats = int(np.ceil(len(clean) / len(noise)))
            noise = np.tile(noise, repeats)[:len(clean)]
        else:
            start = random.randint(0, max(0, len(noise) - len(clean)))
            noise = noise[start:start + len(clean)]
        
        clean_power = np.mean(clean ** 2) + 1e-10
        noise_power = np.mean(noise ** 2) + 1e-10
        
        target_noise_power = clean_power / (10 ** (snr_db / 10))
        scale = np.sqrt(target_noise_power / noise_power)
        
        mixed = clean + noise * scale
        return AudioProcessor.normalize_audio(mixed)
    
    @staticmethod
    def pad_or_trim(audio: np.ndarray, target_length: int) -> np.ndarray:
        """Pad or trim to target length"""
        if len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)))
        else:
            audio = audio[:target_length]
        return audio
    
    @staticmethod
    def random_crop(audio: np.ndarray, crop_length: int) -> np.ndarray:
        """Random crop from audio"""
        if len(audio) <= crop_length:
            return AudioProcessor.pad_or_trim(audio, crop_length)
        start = random.randint(0, len(audio) - crop_length)
        return audio[start:start + crop_length]

audio_processor = AudioProcessor()
print("✓ Audio processor initialized")

✓ Audio processor initialized


In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

print("Hugging Face Authentication Successful!")

Hugging Face Authentication Successful!


In [6]:
# ============================================================================
# PART 5: AST FEATURE PROCESSOR
# ============================================================================

class ASTFeatureProcessor:
    """Process audio to AST-compatible features"""
    
    def __init__(self, sample_rate=16000):
        self.sr = sample_rate
        self.feature_extractor = ASTFeatureExtractor.from_pretrained(
            config.AST_MODEL_NAME,
            sampling_rate=sample_rate
        )
    
    def __call__(self, audio: np.ndarray) -> torch.Tensor:
        """Convert audio to AST input format"""
        inputs = self.feature_extractor(
            audio,
            sampling_rate=self.sr,
            return_tensors="pt"
        )
        return inputs.input_values.squeeze(0)

ast_processor = ASTFeatureProcessor(config.SAMPLE_RATE)
print("✓ AST Feature Processor initialized")


# ============================================================================
# SPECAUGMENT
# ============================================================================

class SpecAugment:
    """SpecAugment for spectrograms"""
    
    def __init__(self, freq_mask_param=20, time_mask_param=40,
                 num_freq_masks=2, num_time_masks=2):
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param
        self.num_freq_masks = num_freq_masks
        self.num_time_masks = num_time_masks
    
    def __call__(self, spec: torch.Tensor) -> torch.Tensor:
        """Apply SpecAugment to spectrogram"""
        if spec.dim() == 2:
            spec = spec.unsqueeze(0)
            squeeze = True
        else:
            squeeze = False
        
        batch, time, freq = spec.shape
        spec = spec.clone()
        
        # Frequency masking
        for _ in range(self.num_freq_masks):
            f = random.randint(1, min(self.freq_mask_param, freq - 1))
            f0 = random.randint(0, freq - f)
            spec[:, :, f0:f0+f] = 0
        
        # Time masking
        for _ in range(self.num_time_masks):
            t = random.randint(1, min(self.time_mask_param, time - 1))
            t0 = random.randint(0, time - t)
            spec[:, t0:t0+t, :] = 0
        
        if squeeze:
            spec = spec.squeeze(0)
        
        return spec

spec_augment = SpecAugment(
    freq_mask_param=config.FREQ_MASK_PARAM,
    time_mask_param=config.TIME_MASK_PARAM,
    num_freq_masks=config.NUM_FREQ_MASKS,
    num_time_masks=config.NUM_TIME_MASKS
)
print("✓ SpecAugment initialized")


# ============================================================================
# DATASET
# ============================================================================

class SyntheticMashupDataset(Dataset):
    """Synthetic mashup dataset with SpecAugment"""
    
    def __init__(self, audio_cache: AudioCache,
                 samples_per_epoch: int = 5000,
                 is_validation: bool = False,
                 apply_augmentation: bool = True):
        
        self.audio_cache = audio_cache
        self.samples_per_epoch = samples_per_epoch
        self.is_validation = is_validation
        self.apply_augmentation = apply_augmentation and not is_validation
        self.sr = config.SAMPLE_RATE
        self.crop_length = config.NUM_SAMPLES
        self.full_length = config.FULL_SAMPLES
        self.spec_augment = spec_augment
    
    def __len__(self):
        return self.samples_per_epoch
    
    def _create_mashup(self, genre: str) -> np.ndarray:
        """Create synthetic mashup"""
        stems = self.audio_cache.get_random_stems_for_genre(genre)
        
        if not stems:
            return np.zeros(self.full_length, dtype=np.float32)
        
        mixed = np.zeros(self.full_length, dtype=np.float32)
        
        for stem_name, stem_audio in stems.items():
            stem_audio = audio_processor.pad_or_trim(stem_audio, self.full_length)
            
            if self.apply_augmentation:
                volume = random.uniform(*config.VOLUME_RANGE)
            else:
                volume = 1.0
            
            mixed += stem_audio * volume
        
        mixed = audio_processor.rms_normalize(mixed, target_db=-20.0)
        
        # Add noise
        if self.apply_augmentation:
            num_noise = random.randint(1, 2)
            for _ in range(num_noise):
                noise = self.audio_cache.get_random_noise()
                snr = random.uniform(*config.SNR_RANGE)
                
                noise_full = np.zeros(self.full_length, dtype=np.float32)
                noise = audio_processor.pad_or_trim(noise, len(noise))
                
                if len(noise) < self.full_length:
                    start = random.randint(0, self.full_length - len(noise))
                    noise_full[start:start + len(noise)] = noise
                else:
                    noise_full = noise[:self.full_length]
                
                mixed = audio_processor.add_noise_snr(mixed, noise_full, snr)
        
        return mixed
    
    def __getitem__(self, idx):
        genre = random.choice(config.GENRES)
        label = config.GENRE2IDX[genre]
        
        audio = self._create_mashup(genre)
        audio_crop = audio_processor.random_crop(audio, self.crop_length)
        
        # Get features
        features = ast_processor(audio_crop)
        
        # Apply SpecAugment
        if self.apply_augmentation and random.random() < config.SPEC_AUGMENT_PROB:
            features = self.spec_augment(features)
        
        label_tensor = torch.zeros(config.NUM_CLASSES, dtype=torch.float32)
        
        # Mixup
        if self.apply_augmentation and random.random() < config.MIXUP_PROB:
            genre2 = random.choice(config.GENRES)
            label2 = config.GENRE2IDX[genre2]
            
            audio2 = self._create_mashup(genre2)
            audio2_crop = audio_processor.random_crop(audio2, self.crop_length)
            features2 = ast_processor(audio2_crop)
            
            if random.random() < config.SPEC_AUGMENT_PROB:
                features2 = self.spec_augment(features2)
            
            lam = np.random.beta(config.MIXUP_ALPHA, config.MIXUP_ALPHA)
            features = lam * features + (1 - lam) * features2
            
            label_tensor[label] = lam
            label_tensor[label2] += (1 - lam)
        else:
            label_tensor[label] = 1.0
        
        return features, label_tensor


# Test dataset
print("\nTesting dataset...")
test_ds = SyntheticMashupDataset(
    audio_cache=audio_cache,
    samples_per_epoch=3,
    is_validation=False
)

for i in range(2):
    features, label = test_ds[i]
    print(f"  Sample {i}: features={features.shape}, label_sum={label.sum():.3f}")

del test_ds
print("✓ Dataset test passed!")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

✓ AST Feature Processor initialized
✓ SpecAugment initialized

Testing dataset...
  Sample 0: features=torch.Size([1024, 128]), label_sum=1.000
  Sample 1: features=torch.Size([1024, 128]), label_sum=1.000
✓ Dataset test passed!


In [7]:
# ============================================================================
# PART 6: AST MODEL
# ============================================================================

class ASTGenreClassifier(nn.Module):
    """Audio Spectrogram Transformer for Genre Classification"""
    
    def __init__(self, num_classes: int = config.NUM_CLASSES,
                 pretrained: bool = True,
                 freeze_encoder: bool = False):
        super().__init__()
        
        print(f"Loading AST model: {config.AST_MODEL_NAME}")
        
        self.ast = ASTModel.from_pretrained(config.AST_MODEL_NAME)
        self.hidden_size = self.ast.config.hidden_size
        
        if freeze_encoder:
            for param in self.ast.parameters():
                param.requires_grad = False
            print("  Encoder frozen")
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.hidden_size),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )
        
        self._init_classifier()
        print(f"  Hidden size: {self.hidden_size}")
    
    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        outputs = self.ast(input_values)
        pooled = outputs.pooler_output
        logits = self.classifier(pooled)
        return logits
    
    def unfreeze_encoder(self):
        for param in self.ast.parameters():
            param.requires_grad = True
        print("Encoder unfrozen")
    
    def freeze_encoder(self):
        for param in self.ast.parameters():
            param.requires_grad = False
        print("Encoder frozen")


# Test model
print("\n" + "="*60)
print("Testing AST Model")
print("="*60)

test_model = ASTGenreClassifier(pretrained=True, freeze_encoder=False).to(device)

# Test forward
print("\n1. Testing forward pass...")
dummy_features = torch.randn(2, 1024, 128).to(device)
with torch.no_grad():
    output = test_model(dummy_features)
    print(f"   Input: {dummy_features.shape} -> Output: {output.shape}")
    print(f"   Contains NaN: {torch.isnan(output).any().item()}")

# Test backward
print("\n2. Testing backward pass...")
dummy_features_2 = torch.randn(2, 1024, 128).to(device)
output_2 = test_model(dummy_features_2)
loss = F.cross_entropy(output_2, torch.randint(0, config.NUM_CLASSES, (2,)).to(device))
loss.backward()
print(f"   Loss: {loss.item():.4f}")
print(f"   Backward: SUCCESS")

# Parameters
total_params = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\n✓ Model test passed!")

del test_model, dummy_features, dummy_features_2, output, output_2, loss
torch.cuda.empty_cache()
gc.collect()


Testing AST Model
Loading AST model: MIT/ast-finetuned-audioset-10-10-0.4593


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Hidden size: 768

1. Testing forward pass...
   Input: torch.Size([2, 1024, 128]) -> Output: torch.Size([2, 10])
   Contains NaN: False

2. Testing backward pass...
   Loss: 2.2069
   Backward: SUCCESS

Total parameters: 86,388,234
Trainable parameters: 86,388,234

✓ Model test passed!


47

In [8]:
# ============================================================================
# PART 7: LOSS FUNCTIONS
# ============================================================================

class SoftTargetCrossEntropy(nn.Module):
    """Cross entropy for soft targets"""
    
    def __init__(self):
        super().__init__()
    
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = F.log_softmax(logits, dim=-1)
        loss = -(targets * log_probs).sum(dim=-1).mean()
        return loss

criterion = SoftTargetCrossEntropy()
print("✓ Loss function initialized")

✓ Loss function initialized


In [9]:
# ============================================================================
# PART 8: TRAINER
# ============================================================================

class Trainer:
    """Trainer with multi-GPU support"""
    
    def __init__(self, model, train_loader, val_loader, device, config):
        self.config = config
        self.device = device
        
        # Multi-GPU
        if config.USE_MULTI_GPU and torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs")
            self.model = nn.DataParallel(model)
        else:
            self.model = model
        
        self.model = self.model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY
        )
        
        # Scheduler
        total_steps = len(train_loader) * config.NUM_EPOCHS
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=config.LEARNING_RATE,
            total_steps=total_steps,
            pct_start=config.WARMUP_RATIO,
            anneal_strategy='cos'
        )
        
        # Loss
        self.criterion = SoftTargetCrossEntropy()
        
        # AMP
        self.scaler = GradScaler() if config.USE_AMP else None
        
        # Tracking
        self.best_f1 = 0.0
        self.best_epoch = 0
        self.train_losses = []
        self.val_losses = []
        self.val_f1_scores = []
        self.patience_counter = 0
    
    def train_epoch(self, epoch: int) -> float:
        self.model.train()
        total_loss = 0
        num_batches = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.config.NUM_EPOCHS}")
        
        for features, targets in pbar:
            features = features.to(self.device)
            targets = targets.to(self.device)
            
            self.optimizer.zero_grad()
            
            if self.config.USE_AMP:
                with autocast():
                    logits = self.model(features)
                    loss = self.criterion(logits, targets)
                
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.GRADIENT_CLIP)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                logits = self.model(features)
                loss = self.criterion(logits, targets)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.GRADIENT_CLIP)
                self.optimizer.step()
            
            self.scheduler.step()
            total_loss += loss.item()
            num_batches += 1
            
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'avg': f'{total_loss/num_batches:.4f}',
                'lr': f'{self.scheduler.get_last_lr()[0]:.2e}'
            })
        
        return total_loss / num_batches
    
    def validate(self) -> Tuple[float, float, np.ndarray]:
        self.model.eval()
        total_loss = 0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for features, targets in tqdm(self.val_loader, desc="Validating"):
                features = features.to(self.device)
                targets = targets.to(self.device)
                
                if self.config.USE_AMP:
                    with autocast():
                        logits = self.model(features)
                        loss = self.criterion(logits, targets)
                else:
                    logits = self.model(features)
                    loss = self.criterion(logits, targets)
                
                total_loss += loss.item()
                
                preds = torch.argmax(logits, dim=1)
                true_labels = torch.argmax(targets, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(true_labels.cpu().numpy())
        
        avg_loss = total_loss / len(self.val_loader)
        macro_f1 = f1_score(all_targets, all_preds, average='macro')
        per_class_f1 = f1_score(all_targets, all_preds, average=None)
        
        return avg_loss, macro_f1, per_class_f1
    
    def save_checkpoint(self, epoch: int, f1_score: float, path: Path):
        model_state = self.model.module.state_dict() if hasattr(self.model, 'module') else self.model.state_dict()
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model_state,
            'optimizer_state_dict': self.optimizer.state_dict(),
            'f1_score': f1_score,
        }
        torch.save(checkpoint, path)
    
    def train(self) -> float:
        print(f"\n{'='*60}")
        print(f"Starting training for {self.config.NUM_EPOCHS} epochs")
        print(f"Training samples: {len(self.train_loader.dataset)}")
        print(f"Validation samples: {len(self.val_loader.dataset)}")
        print(f"{'='*60}\n")
        
        for epoch in range(self.config.NUM_EPOCHS):
            train_loss = self.train_epoch(epoch)
            self.train_losses.append(train_loss)
            
            val_loss, val_f1, per_class_f1 = self.validate()
            self.val_losses.append(val_loss)
            self.val_f1_scores.append(val_f1)
            
            print(f"\nEpoch {epoch+1}/{self.config.NUM_EPOCHS}")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"Val Macro F1: {val_f1:.4f}")
            print("Per-class F1:", end=" ")
            for i, f1 in enumerate(per_class_f1):
                print(f"{config.GENRES[i][:3]}:{f1:.3f}", end=" ")
            print()
            
            if val_f1 > self.best_f1:
                self.best_f1 = val_f1
                self.best_epoch = epoch
                self.patience_counter = 0
                self.save_checkpoint(epoch, val_f1, config.OUTPUT_DIR / 'best_model.pth')
                print(f"✓ New best F1: {val_f1:.4f}")
            else:
                self.patience_counter += 1
                print(f"No improvement. Patience: {self.patience_counter}/{self.config.PATIENCE}")
            
            if self.patience_counter >= self.config.PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
        
        print(f"\n{'='*60}")
        print(f"Training complete! Best F1: {self.best_f1:.4f} at epoch {self.best_epoch+1}")
        print(f"{'='*60}")
        
        return self.best_f1

In [10]:
# ============================================================================
# PART 9: DATA LOADERS AND ENSEMBLE TRAINING
# ============================================================================

def create_data_loaders():
    """Create train and validation data loaders"""
    train_dataset = SyntheticMashupDataset(
        audio_cache=audio_cache,
        samples_per_epoch=config.SYNTHETIC_SAMPLES_PER_EPOCH,
        is_validation=False,
        apply_augmentation=True
    )
    
    val_dataset = SyntheticMashupDataset(
        audio_cache=audio_cache,
        samples_per_epoch=config.VAL_SAMPLES,
        is_validation=True,
        apply_augmentation=False
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
    
    return train_loader, val_loader


def train_ensemble(num_models: int = 3) -> List[Path]:
    """Train ensemble of models"""
    
    print(f"\n{'='*60}")
    print(f"TRAINING ENSEMBLE OF {num_models} MODELS")
    print(f"{'='*60}")
    
    ensemble_paths = []
    
    for model_idx in range(num_models):
        print(f"\n{'='*60}")
        print(f"TRAINING MODEL {model_idx + 1}/{num_models}")
        print(f"{'='*60}")
        
        # Different seed for each model
        set_seed(42 + model_idx * 111)
        
        # Fresh data loaders
        train_loader, val_loader = create_data_loaders()
        
        # Create model
        model = ASTGenreClassifier(
            num_classes=config.NUM_CLASSES,
            pretrained=True,
            freeze_encoder=False
        )
        
        # Train
        trainer = Trainer(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            config=config
        )
        
        best_f1 = trainer.train()
        
        # Save
        model_path = config.OUTPUT_DIR / f'model_{model_idx}.pth'
        shutil.copy(config.OUTPUT_DIR / 'best_model.pth', model_path)
        ensemble_paths.append(model_path)
        
        print(f"\n✓ Model {model_idx + 1} saved: {model_path} (F1: {best_f1:.4f})")
        
        # Cleanup
        del model, trainer, train_loader, val_loader
        torch.cuda.empty_cache()
        gc.collect()
    
    print(f"\n{'='*60}")
    print(f"ENSEMBLE TRAINING COMPLETE!")
    print(f"Trained {len(ensemble_paths)} models")
    print(f"{'='*60}")
    
    return ensemble_paths


# Train ensemble
ensemble_paths = train_ensemble(num_models=config.NUM_ENSEMBLE_MODELS)


TRAINING ENSEMBLE OF 3 MODELS

TRAINING MODEL 1/3
Loading AST model: MIT/ast-finetuned-audioset-10-10-0.4593


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Hidden size: 768
Using 2 GPUs

Starting training for 25 epochs
Training samples: 8000
Validation samples: 1500



Epoch 1/25:   0%|          | 0/167 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/32 [00:00<?, ?it/s]


Epoch 24/25
Train Loss: 0.1662 | Val Loss: 0.0370
Val Macro F1: 0.9897
Per-class F1: blu:0.987 cla:1.000 cou:0.987 dis:0.989 hip:0.993 jaz:0.990 met:0.996 pop:0.984 reg:0.990 roc:0.979 
No improvement. Patience: 6/10


Epoch 25/25:   0%|          | 0/167 [00:00<?, ?it/s]

Validating:   0%|          | 0/32 [00:00<?, ?it/s]


Epoch 25/25
Train Loss: 0.1605 | Val Loss: 0.0337
Val Macro F1: 0.9870
Per-class F1: blu:0.994 cla:0.997 cou:0.983 dis:0.993 hip:0.985 jaz:0.997 met:0.988 pop:0.989 reg:0.991 roc:0.954 
No improvement. Patience: 7/10

Training complete! Best F1: 0.9932 at epoch 18

✓ Model 3 saved: /kaggle/working/model_2.pth (F1: 0.9932)

ENSEMBLE TRAINING COMPLETE!
Trained 3 models


In [11]:
# ============================================================================
# PART 10: INFERENCE WITH ENSEMBLE AND TTA
# ============================================================================

class TestDataset(Dataset):
    """Test dataset with multiple TTA crops"""
    
    def __init__(self, mashup_dir: Path, test_csv: pd.DataFrame, n_crops: int = 9):
        self.mashup_dir = mashup_dir
        self.test_df = test_csv
        self.n_crops = n_crops
        self.sr = config.SAMPLE_RATE
        self.crop_length = config.NUM_SAMPLES
        self.full_length = config.FULL_SAMPLES
    
    def __len__(self):
        return len(self.test_df)
    
    def _load_audio(self, audio_id) -> np.ndarray:
        filename = f"song{str(audio_id).zfill(4)}.wav"
        path = self.mashup_dir / filename
        
        try:
            audio, _ = librosa.load(path, sr=self.sr, duration=30, mono=True)
            audio = audio_processor.pad_or_trim(audio, self.full_length)
            audio = audio_processor.normalize_audio(audio)
            return audio.astype(np.float32)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return np.zeros(self.full_length, dtype=np.float32)
    
    def _create_crops(self, audio: np.ndarray) -> List[torch.Tensor]:
        crops = []
        total = len(audio)
        crop_len = self.crop_length
        
        if total <= crop_len:
            audio = audio_processor.pad_or_trim(audio, crop_len)
            features = ast_processor(audio)
            return [features] * self.n_crops
        
        max_start = total - crop_len
        starts = np.linspace(0, max_start, self.n_crops, dtype=int)
        
        for start in starts:
            crop = audio[start:start + crop_len]
            features = ast_processor(crop)
            crops.append(features)
        
        return crops
    
    def __getitem__(self, idx):
        row = self.test_df.iloc[idx]
        audio_id = row['id']
        
        audio = self._load_audio(audio_id)
        crops = self._create_crops(audio)
        crops_tensor = torch.stack(crops)
        
        return str(audio_id), crops_tensor


def ensemble_predict_with_tta(models: List[nn.Module], test_loader, device) -> Tuple[Dict, Dict]:
    """Ensemble prediction with TTA"""
    
    for model in models:
        model.eval()
    
    predictions = {}
    all_probs = {}
    
    with torch.no_grad():
        for audio_ids, crops_batch in tqdm(test_loader, desc="Ensemble Predicting"):
            batch_size, n_crops = crops_batch.shape[:2]
            crops_flat = crops_batch.view(-1, *crops_batch.shape[2:]).to(device)
            
            ensemble_probs = []
            
            for model in models:
                if config.USE_AMP:
                    with autocast():
                        logits = model(crops_flat)
                else:
                    logits = model(crops_flat)
                
                probs = F.softmax(logits, dim=1)
                probs = probs.view(batch_size, n_crops, -1)
                avg_probs = probs.mean(dim=1)
                ensemble_probs.append(avg_probs)
            
            final_probs = torch.stack(ensemble_probs).mean(dim=0)
            preds = torch.argmax(final_probs, dim=1)
            
            for i, audio_id in enumerate(audio_ids):
                predictions[audio_id] = config.IDX2GENRE[preds[i].item()]
                all_probs[audio_id] = final_probs[i].cpu().numpy()
    
    return predictions, all_probs


# Load test data
print("\n" + "="*60)
print("INFERENCE")
print("="*60)

print("\nLoading test data...")
test_df = pd.read_csv(config.TEST_CSV)
print(f"Test samples: {len(test_df)}")

# Create test loader
test_dataset = TestDataset(
    mashup_dir=config.MASHUPS_DIR,
    test_csv=test_df,
    n_crops=config.TTA_CROPS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=6,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Load ensemble models
print("\nLoading ensemble models...")
models = []

for model_path in ensemble_paths:
    model = ASTGenreClassifier(
        num_classes=config.NUM_CLASSES,
        pretrained=False
    )
    
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    models.append(model)
    print(f"  ✓ Loaded: {model_path} (F1: {checkpoint['f1_score']:.4f})")

print(f"\nEnsemble size: {len(models)} models")
print(f"TTA crops: {config.TTA_CROPS}")

# Generate predictions
print("\nGenerating ensemble predictions...")
predictions, all_probs = ensemble_predict_with_tta(models, test_loader, device)

# Create submission
submission_df = pd.DataFrame([
    {'id': aid, 'genre': genre}
    for aid, genre in predictions.items()
])

submission_df['id'] = submission_df['id'].astype(int)
submission_df = submission_df.sort_values('id')
submission_df['id'] = submission_df['id'].astype(str).str.zfill(4)

# Save
submission_path = config.OUTPUT_DIR / 'submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"\n{'='*60}")
print("INFERENCE COMPLETE!")
print(f"{'='*60}")
print(f"✓ Saved to: {submission_path}")
print(f"Total predictions: {len(submission_df)}")

print("\nGenre distribution:")
print(submission_df['genre'].value_counts().sort_index())

# Distribution stats
counts = submission_df['genre'].value_counts()
print(f"\nDistribution stats:")
print(f"  Min: {counts.min()} ({counts.idxmin()})")
print(f"  Max: {counts.max()} ({counts.idxmax()})")
print(f"  Std: {counts.std():.1f}")
print(f"  Expected: {len(submission_df) / 10:.0f}")


INFERENCE

Loading test data...
Test samples: 3020

Loading ensemble models...
Loading AST model: MIT/ast-finetuned-audioset-10-10-0.4593


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Hidden size: 768
  ✓ Loaded: /kaggle/working/model_0.pth (F1: 0.9930)
Loading AST model: MIT/ast-finetuned-audioset-10-10-0.4593


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Hidden size: 768
  ✓ Loaded: /kaggle/working/model_2.pth (F1: 0.9932)

Ensemble size: 3 models
TTA crops: 9

Generating ensemble predictions...


Ensemble Predicting:   0%|          | 0/504 [00:00<?, ?it/s]


INFERENCE COMPLETE!
✓ Saved to: /kaggle/working/submission.csv
Total predictions: 3020

Genre distribution:
genre
blues        314
classical    264
country      264
disco        307
hiphop       311
jazz         297
metal        318
pop          313
reggae       311
rock         321
Name: count, dtype: int64

Distribution stats:
  Min: 264 (classical)
  Max: 321 (rock)
  Std: 21.0
  Expected: 302


In [12]:
# ============================================================================
# PART 11: ANALYSIS
# ============================================================================

def analyze_predictions(predictions: Dict, test_df: pd.DataFrame):
    """Analyze prediction distribution"""
    
    print("\n" + "="*60)
    print("PREDICTION ANALYSIS")
    print("="*60)
    
    # Genre counts
    genre_counts = {}
    for genre in config.GENRES:
        genre_counts[genre] = sum(1 for v in predictions.values() if v == genre)
    
    print("\nGenre Distribution:")
    print("-" * 40)
    for genre in config.GENRES:
        count = genre_counts[genre]
        pct = count / len(predictions) * 100
        bar = "█" * int(pct * 2)
        print(f"{genre:12s}: {count:4d} ({pct:5.1f}%) {bar}")
    
    # Balance check
    expected = len(predictions) / 10
    imbalance = max(genre_counts.values()) - min(genre_counts.values())
    print(f"\nBalance check:")
    print(f"  Expected per class: {expected:.0f}")
    print(f"  Actual range: {min(genre_counts.values())} - {max(genre_counts.values())}")
    print(f"  Imbalance: {imbalance}")

analyze_predictions(predictions, test_df)


def save_probability_file(all_probs: Dict, path: Path):
    """Save probabilities for potential post-processing"""
    
    prob_df = pd.DataFrame([
        {'id': aid, **{config.GENRES[i]: probs[i] for i in range(len(probs))}}
        for aid, probs in all_probs.items()
    ])
    
    prob_df['id'] = prob_df['id'].astype(int)
    prob_df = prob_df.sort_values('id')
    prob_df['id'] = prob_df['id'].astype(str).str.zfill(4)
    
    prob_df.to_csv(path, index=False)
    print(f"\n✓ Probabilities saved to: {path}")

save_probability_file(all_probs, config.OUTPUT_DIR / 'probabilities.csv')


print("\n" + "="*60)
print("ALL DONE!")
print("="*60)
print(f"Submission file: {submission_path}")
print(f"Probabilities file: {config.OUTPUT_DIR / 'probabilities.csv'}")
print("\nGood luck! 🎵")


PREDICTION ANALYSIS

Genre Distribution:
----------------------------------------
blues       :  314 ( 10.4%) ████████████████████
classical   :  264 (  8.7%) █████████████████
country     :  264 (  8.7%) █████████████████
disco       :  307 ( 10.2%) ████████████████████
hiphop      :  311 ( 10.3%) ████████████████████
jazz        :  297 (  9.8%) ███████████████████
metal       :  318 ( 10.5%) █████████████████████
pop         :  313 ( 10.4%) ████████████████████
reggae      :  311 ( 10.3%) ████████████████████
rock        :  321 ( 10.6%) █████████████████████

Balance check:
  Expected per class: 302
  Actual range: 264 - 321
  Imbalance: 57

✓ Probabilities saved to: /kaggle/working/probabilities.csv

ALL DONE!
Submission file: /kaggle/working/submission.csv
Probabilities file: /kaggle/working/probabilities.csv

Good luck! 🎵
